In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteBayesianEstimator
from pgmpy.inference import VariableElimination

## Step 0 — Load and Explore the Dataset

In [48]:
df = pd.read_excel('dataset_exercise.xlsx')

print(f"Total rows        : {df.shape[0]}")
print(f"Total columns     : {df.shape[1]}")
print(f"Unique alarms     : {df['alarm_id'].nunique()}")
print(f"Unique windows    : {df['time_window'].nunique()}")
print(f"Window range      : {df['time_window'].min()} → {df['time_window'].max()}")
print(f"Machine states    : {df['machine_state'].unique().tolist()}")
print(df.isnull().sum())
print(df.head())
print(df.tail())

Total rows        : 4307
Total columns     : 5
Unique alarms     : 94
Unique windows    : 329
Window range      : 1 → 837
Machine states    : ['Failure', 'Running']
start_alarm      0
end_alarm        5
alarm_id         0
machine_state    0
time_window      0
dtype: int64
              start_alarm               end_alarm   alarm_id machine_state  \
0 2025-06-23 03:08:36.666 2025-06-23 03:08:38.663  156700113       Failure   
1 2025-06-23 03:08:36.666 2025-06-23 03:08:38.663  156700114       Failure   
2 2025-06-23 03:08:42.154 2025-06-23 03:10:12.667  156701901       Failure   
3 2025-06-23 03:08:42.669 2025-06-23 03:10:12.667  156702002       Failure   
4 2025-06-23 03:08:42.669 2025-06-23 03:10:12.667  156701902       Failure   

   time_window  
0            1  
1            1  
2            1  
3            1  
4            1  
                 start_alarm               end_alarm   alarm_id machine_state  \
4302 2025-07-04 17:39:21.725 2025-07-04 17:45:20.718  156701909       Failu

In [50]:
# Alarms per window
print(df.groupby('time_window')['alarm_id'].count().describe())

# State distribution per window
state_per_window = df.groupby('time_window')['machine_state'].first()
print(state_per_window.value_counts())

# Mixed state windows
mixed = df.groupby('time_window')['machine_state'].nunique()
print(f"Windows with mixed states: {(mixed > 1).sum()}")

# Duration distribution
df['duration_sec'] = (pd.to_datetime(df['end_alarm']) - pd.to_datetime(df['start_alarm'])).dt.total_seconds()
print(df['duration_sec'].describe())

count    329.000000
mean      13.091185
std       14.156278
min        1.000000
25%        5.000000
50%        8.000000
75%       16.000000
max      107.000000
Name: alarm_id, dtype: float64
machine_state
Running    195
Failure    134
Name: count, dtype: int64
Windows with mixed states: 35
count     4302.000000
mean       206.655719
std       1893.054250
min          0.471000
25%          2.006000
50%         14.501500
75%         75.492000
max      48936.207000
Name: duration_sec, dtype: float64


### Key Findings

| Finding | Impact |

| 4307 rows, one per alarm event | Must summarize into one row per window |
| 94 unique alarm types | Too many for DBN — alarm selection needed |
| 329 unique windows, range 1→837 | 508 gaps exist — quiet periods with no alarms |
| 5 missing end_alarm values | Must handle before duration calculation |
| 35 windows with mixed states | Need majority vote for one label per window |
| 75% of durations under 75 seconds | Exercise thresholds (30,300) would skew data |
| Max duration 48,936 seconds | Valid outlier — maps to Long category |
| Running: 195, Failure: 134 windows | Mild imbalance — recall is primary metric |

## Step 0b — Outlier 
There is an outlier when the alarm is active for 13.5hrs but it is not an error but it is considered as a critical fault that persisted for an extended period. Since, all durations are  discretize into categories, this maps to Long which is same eas any value above 75seconds. 

In [51]:
max_idx = df['duration_sec'].idxmax()
outlier = df.loc[max_idx, ['alarm_id', 'time_window', 'machine_state', 'duration_sec']]
print("Outlier details:")
print(outlier)

Outlier details:
alarm_id         156701900
time_window            391
machine_state      Failure
duration_sec     48936.207
Name: 2173, dtype: object


## Step 1 — Preprocessing

**Fix timestamps** — convert to datetime so we can subtract end_alarm − start_alarm.

**Handle 5 missing end_alarm** — fill with start_alarm so duration = 0 → None category.

**Majority vote for mixed windows** — 35 windows have both Running and Failure rows. 
DBN needs one label per window. Ties go to Failure — missing a real failure is more 
costly than a false alarm.

**Adjusted duration thresholds** — 75% of durations are under 75 seconds. Original 
thresholds (30, 300) would put almost everything into Short, making duration useless. 
Using 10 and 75 based on actual data percentiles, which gives balanced categories 
(Long=40%, Short=31%, Medium=29%).

In [52]:
# Fix timestamps
df['start_alarm'] = pd.to_datetime(df['start_alarm'])
df['end_alarm']   = pd.to_datetime(df['end_alarm'])

# Handle 5 missing end_alarm
df['end_alarm'] = df['end_alarm'].fillna(df['start_alarm'])

# Compute duration in seconds
df['duration_sec'] = (df['end_alarm'] - df['start_alarm']).dt.total_seconds()

# Majority vote — one state per window, ties go to Failure
def majority_state(states):
    counts = states.value_counts()
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        return 'Failure'
    return counts.index[0]

window_state = (
    df.groupby('time_window')['machine_state']
    .apply(majority_state)
    .reset_index()
    .rename(columns={'machine_state': 'state'})
)

# Count and duration per (window, alarm)
alarm_count = df.groupby(['time_window', 'alarm_id']).size().reset_index(name='count')
alarm_dur   = df.groupby(['time_window', 'alarm_id'])['duration_sec'].sum().reset_index()

# Discretize count (exercise thresholds)
def cat_count(c):
    if c == 0:  return 'None'
    if 1 <= c <= 2: return 'Low'
    if 3 <= c <= 5:  return 'Medium'
    return 'High'

# Discretize duration (adjusted based on actual data distribution)
def cat_duration(d):
    if d == 0:   return 'None'
    if d <= 10:  return 'Short'
    if d <= 75:  return 'Medium'
    return 'Long'

alarm_count['count_cat']  = alarm_count['count'].apply(cat_count)
alarm_dur['duration_cat'] = alarm_dur['duration_sec'].apply(cat_duration)

# Pivot to wide format
pivot_count = alarm_count.pivot(
    index='time_window', columns='alarm_id', values='count_cat'
).fillna('None')
pivot_dur = alarm_dur.pivot(
    index='time_window', columns='alarm_id', values='duration_cat'
).fillna('None')

pivot_count.columns = [f'A{c}_count'    for c in pivot_count.columns]
pivot_dur.columns   = [f'A{c}_duration' for c in pivot_dur.columns]

windows_df = pivot_count.join(pivot_dur).merge(window_state, on='time_window')

print(f"Windows : {windows_df.shape[0]}")
print(f"Features: {windows_df.shape[1] - 1}")
print(windows_df['state'].value_counts())

Windows : 329
Features: 189
state
Running    207
Failure    122
Name: count, dtype: int64


### Preprocessing Validation

In [54]:
# Verify majority vote changed labels meaningfully
original = df.groupby('time_window')['machine_state'].first().reset_index()
original.columns = ['time_window', 'original_state']
comp = window_state.merge(original, on='time_window')
changed = (comp['state'] != comp['original_state']).sum()
print(f"Windows where majority vote changed the label: {changed}")

# Verify adjusted thresholds produced balanced categories
print("\nDuration category distribution with adjusted thresholds:")
print(alarm_dur['duration_cat'].value_counts(normalize=True))

Windows where majority vote changed the label: 18

Duration category distribution with adjusted thresholds:
duration_cat
Long      0.398871
Short     0.310575
Medium    0.289528
None      0.001027
Name: proportion, dtype: float64


## Step 2 — Generate Training Transitions

A DBN learns by observing how the system evolves from one time step to the next. 
To capture this temporal evolution, each time window t needs to pair with the 
following window t+1, creating transition rows of the form:

(State_t, AlarmFeatures_t) → State_t+1

We first split our observed windows into training (80%) and testing (20%) 
sets chronologically — earlier windows for training, later windows for 
testing. This preserves the temporal order of the data.

We then reconstruct the complete window sequence independently for each 
split by inserting synthetic windows for quiet periods where no alarms 
were recorded. Each synthetic window is assigned all alarm features = None 
and state = Running, correctly capturing that silence indicates normal 
machine operation.

In [55]:
# Chronological train/test split on observed windows
split_idx     = int(len(windows_df) * 0.8)
train_windows = windows_df.iloc[:split_idx]
test_windows  = windows_df.iloc[split_idx:]

def reconstruct_gaps(windows_subset):
    all_w = pd.DataFrame({'time_window': range(
        windows_subset['time_window'].min(),
        windows_subset['time_window'].max() + 1
    )})
    full = all_w.merge(windows_subset, on='time_window', how='left')
    for col in [c for c in windows_subset.columns if c != 'time_window']:
        if col == 'state': full[col] = full[col].fillna('Running')
        else:              full[col] = full[col].fillna('None')
    return full.sort_values('time_window').reset_index(drop=True)

train_full = reconstruct_gaps(train_windows)
test_full  = reconstruct_gaps(test_windows)

def make_transitions(full_df):
    t = full_df.copy()
    t['state_next'] = t['state'].shift(-1)
    return t.dropna(subset=['state_next'])

train_transitions = make_transitions(train_full)
test_transitions  = make_transitions(test_full)

print(f"Train transitions : {len(train_transitions)}")
print(f"Test  transitions : {len(test_transitions)}")
print(f"Train Failure     : {(train_transitions['state_next']=='Failure').sum()}")
print(f"Test  Failure     : {(test_transitions['state_next']=='Failure').sum()}")

Train transitions : 668
Test  transitions : 167
Train Failure     : 82
Test  Failure     : 39


## Step 3 — Alarm Selection

With 94 alarms × 2 features = 188 features, the CPT would have too many combinations to estimate from our limited data. We select the most informative alarms using two independent statistical methods — Chi-Square and Mutual Information — both evaluated against state_next (the future state we want to predict). Feature selection is performed on training data only to prevent leakage into the test set.

In [56]:
feature_cols_only = [c for c in windows_df.columns
                     if c not in ['time_window', 'state']]

chi2_results = []
for col in feature_cols_only:
    contingency = pd.crosstab(
        train_transitions[col], train_transitions['state_next'])
    if contingency.shape[0] < 2: continue
    chi2, p, dof, expected = chi2_contingency(contingency)
    chi2_results.append({'feature': col, 'chi2': round(chi2,3), 'p_value': round(p,6)})

chi2_df = pd.DataFrame(chi2_results).sort_values('chi2', ascending=False).reset_index(drop=True)
chi2_df['chi2_rank'] = chi2_df.index + 1

le = LabelEncoder()
X = train_transitions[feature_cols_only].apply(lambda col: le.fit_transform(col.astype(str)))
y = le.fit_transform(train_transitions['state_next'])
mi_scores = mutual_info_classif(X, y, discrete_features=True, random_state=42)
mi_df = pd.DataFrame({'feature': feature_cols_only,
                      'mutual_info': np.round(mi_scores, 4)
                     }).sort_values('mutual_info', ascending=False).reset_index(drop=True)
mi_df['mi_rank'] = mi_df.index + 1

comparison = chi2_df.merge(mi_df, on='feature')
comparison['rank_difference'] = abs(comparison['chi2_rank'] - comparison['mi_rank'])
comparison = comparison.sort_values('chi2_rank')

print(comparison[['feature','chi2','chi2_rank',
                  'mutual_info','mi_rank','rank_difference']].head(15).to_string(index=False))

agreed = comparison[comparison['rank_difference'] <= 3].head(10)
top_alarm_ids = set()
for f in agreed['feature'].tolist():
    alarm_id = f.split('_count')[0].split('_duration')[0].replace('A','')
    top_alarm_ids.add(alarm_id)

print(f"\nSelected alarms: {top_alarm_ids}")

            feature   chi2  chi2_rank  mutual_info  mi_rank  rank_difference
A156701922_duration 32.463          1       0.0145        2                1
   A156701922_count 31.534          2       0.0140        3                1
A156701900_duration 30.686          3       0.0181        1                2
   A156702002_count 19.385          4       0.0120        7                3
A156701909_duration 19.295          5       0.0128        4                1
   A156701900_count 18.576          6       0.0111       12                6
A156702002_duration 18.420          7       0.0111       13                6
A156701901_duration 18.383          8       0.0124        5                3
A156701902_duration 18.191          9       0.0123        6                3
   A156700113_count 17.447         10       0.0118        9                1
   A156700114_count 17.447         11       0.0118        8                3
A156700113_duration 16.920         12       0.0116       11                1

## Step 4 — Define DBN Structure

Following the exercise specification, the future machine state at t+1 
depends on the current machine state and the alarm features observed at 
time t. To keep the CPT estimable from our available data, we use the 
top 2 most informative alarms, giving 2 × 4^4 = 512 combinations — 
fully learnable from our training transitions.

DBN edges:
- State_t → State_t+1
- Each alarm feature at time t → State_t+1

In [57]:
# Use top 2 alarms dynamically from alarm selection
top_2_alarms = sorted(list(top_alarm_ids))[:2]
print(f"Top 2 alarms: {top_2_alarms}")
print(f"CPT combinations: 2 x 4^4 = {2 * 4**4}")

dynamic_features = []
for alarm in top_2_alarms:
    dynamic_features.extend([f'A{alarm}_count', f'A{alarm}_duration'])

dbn_columns = ['state', 'state_next'] + dynamic_features
train_df = train_transitions[dbn_columns].copy()
test_df  = test_transitions[dbn_columns].copy()

rename_map = {'state': 'State_t', 'state_next': 'State_t+1'}
for col in dynamic_features:
    rename_map[col] = f'{col}_t'

train_df = train_df.rename(columns=rename_map)
test_df  = test_df.rename(columns=rename_map)
feature_t_cols = [rename_map[c] for c in dynamic_features]

edges = [('State_t', 'State_t+1')]
for col in feature_t_cols:
    edges.append((col, 'State_t+1'))

print("\nDBN Structure:")
for edge in edges:
    print(f"  {edge[0]} → {edge[1]}")

Top 2 alarms: ['156700113', '156700114']
CPT combinations: 2 x 4^4 = 512

DBN Structure:
  State_t → State_t+1
  A156700113_count_t → State_t+1
  A156700113_duration_t → State_t+1
  A156700114_count_t → State_t+1
  A156700114_duration_t → State_t+1


## Step 5 — Learn Conditional Probability Tables (CPTs)

The CPT stores the probability of each next state for every combination 
of parent values. We use BDeu Bayesian estimation with equivalent sample 
size of 1 — this applies minimal Laplace smoothing to prevent zero 
probabilities for combinations not seen in training data.

In [58]:
model = DiscreteBayesianNetwork(edges)
est = DiscreteBayesianEstimator(prior_type='BDeu', equivalent_sample_size=1)
est = est.fit(model, train_df)
for cpd in est.parameters_:
    model.add_cpds(cpd)

print(f"Model valid : {model.check_model()}")
print(f"CPDs learned: {len(model.cpds)}")

# Sample predictions
inference = VariableElimination(model)

evidence_silent = {col: 'None' for col in feature_t_cols}
evidence_silent['State_t'] = 'Running'
result = inference.query(variables=['State_t+1'],
                         evidence=evidence_silent, show_progress=False)
print("\nRunning machine, all alarms silent:")
print(result)

Model valid : True
CPDs learned: 6

Running machine, all alarms silent:
+--------------------+------------------+
| State_t+1          |   phi(State_t+1) |
+====================+==================+
| State_t+1(Failure) |           0.0690 |
+--------------------+------------------+
| State_t+1(Running) |           0.9310 |
+--------------------+------------------+


## Step 6 — Evaluation

We evaluate the model on the held-out test set. Since missing a real 
failure is far more costly than a false alarm in predictive maintenance, 
recall is our primary metric. We also evaluate across different 
probability thresholds to find the best operating point.

In [59]:
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, classification_report)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

def predict_threshold(test_df, threshold=0.5):
    y_true, y_pred = [], []
    for _, row in test_df.iterrows():
        evidence = {col: row[col] for col in feature_t_cols}
        evidence['State_t'] = row['State_t']
        try:
            result = inference.query(variables=['State_t+1'],
                                     evidence=evidence, show_progress=False)
            sn   = result.state_names['State_t+1']
            vl   = result.values
            prob = dict(zip(sn, vl))
            pred = 'Failure' if prob.get('Failure', 0) > threshold else 'Running'
        except Exception:
            pred = 'Running'
        y_pred.append(pred)
        y_true.append(row['State_t+1'])
    return y_true, y_pred

y_true, y_pred = predict_threshold(test_df, 0.5)

print("Results at threshold = 0.5")
print(f"Accuracy  : {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision : {precision_score(y_true, y_pred, pos_label='Failure', zero_division=0):.3f}")
print(f"Recall    : {recall_score(y_true, y_pred, pos_label='Failure', zero_division=0):.3f}")
print(f"F1        : {f1_score(y_true, y_pred, pos_label='Failure', zero_division=0):.3f}")
print()
print(classification_report(y_true, y_pred, zero_division=0))

print(f"\n{'Threshold':>10} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("=" * 55)
for t in [0.1, 0.2, 0.3, 0.4, 0.5]:
    yt, yp = predict_threshold(test_df, t)
    print(f"{t:>10.1f} "
          f"{accuracy_score(yt,yp):>10.3f} "
          f"{precision_score(yt,yp,pos_label='Failure',zero_division=0):>10.3f} "
          f"{recall_score(yt,yp,pos_label='Failure',zero_division=0):>10.3f} "
          f"{f1_score(yt,yp,pos_label='Failure',zero_division=0):>10.3f}")

cm = confusion_matrix(y_true, y_pred, labels=['Failure','Running'])
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Failure','Running'],
            yticklabels=['Failure','Running'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — DBN Failure Prediction')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nConfusion matrix saved.")

Results at threshold = 0.5
Accuracy  : 0.778
Precision : 0.625
Recall    : 0.128
F1        : 0.213

              precision    recall  f1-score   support

     Failure       0.62      0.13      0.21        39
     Running       0.79      0.98      0.87       128

    accuracy                           0.78       167
   macro avg       0.71      0.55      0.54       167
weighted avg       0.75      0.78      0.72       167


 Threshold   Accuracy  Precision     Recall         F1
       0.1      0.701      0.407      0.615      0.490
       0.2      0.814      0.605      0.590      0.597
       0.3      0.814      0.605      0.590      0.597
       0.4      0.814      0.605      0.590      0.597
       0.5      0.778      0.625      0.128      0.213

Confusion matrix saved.
